# IR Pipeline — Colab launcher

**О чём ноутбук:** клонирование репозитория (`git clone`), установка пакета, загрузка **готового** датасета с Hugging Face без Drive (`fetch-data`), затем обучение RandomForest:

- на **GPU** — после `pip install -e ".[cuml]"` и включённого Runtime → GPU (библиотека **cuML**);
- на **обычном железе (CPU)** — только **sklearn**, переменная `IR_RF_BACKEND=sklearn`.

Датасет по умолчанию на HF: **Lamblador/IRSpectra2** (см. CLI `--repo-id` при необходимости).

Если исходные JCAMP и сборка через Drive нужны вместо HF — см. `configs/paths.colab.yaml`.

In [ ]:
# Шаг 1: клонирование репозитория и установка пакета (CPU-базовые зависимости)
import subprocess
from pathlib import Path

# from google.colab import drive
# drive.mount("/content/drive")

# Замените URL на свой форк или основной репозиторий, например:
# REPO_URL = "https://github.com/Lamblador/IR_expert_system_3.git"
REPO_URL = "https://github.com/YOUR_USER/IR_expert_system_3.git"
REPO_DIR = Path("IR_expert_system_3")

if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"{REPO_DIR}: уже есть — clone пропущен (pull делайте вручную при необходимости).")

%cd IR_expert_system_3

!pip install -q -e .

### Hugging Face: импорт готового датасета

Команда `fetch-data` скачивает zip из файлов Dataset repo (по умолчанию `Lamblador/IRSpectra2`) и при необходимости распаковывает его в `data/processed/`.

Для **приватного** или gated-репозитория задайте токен чтения (секрет Colab или вручную).

In [ ]:
# Шаг 2: загрузка с Hugging Face (dataset_mini совпадает с configs/paths.huggingface.yaml)
import os

# from google.colab import userdata
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

os.makedirs("data/processed", exist_ok=True)

!ir-pipeline fetch-data --filename dataset_mini.zip --extract-to data/processed

# Полный собранный датасет (раскомментируйте; затем в train используйте --dataset-version dataset_v001 или отдельный yaml):
# !ir-pipeline fetch-data --filename dataset_v001.zip --extract-to data/processed

### Обучение на GPU (NVIDIA + cuML)

1. **Runtime → Change runtime type → Hardware accelerator → GPU**
2. Выполните ячейку: ставится optional-extra **`cuml`**, затем `train`. При `rf_backend`=`auto` (по умолчанию в конфиге) будет выбран **cuML RandomForest**, если CUDA доступна.

Если установка `cuml-cu12` не поддерживается на вашем runtime — используйте блок CPU ниже.

In [ ]:
# Шаг 3a: обучение на GPU (cuML Random Forest)
import os

# После запуска CPU-блока сбросьте переменную, чтобы снова был режим auto:
os.environ.pop("IR_RF_BACKEND", None)

# Явно заставить только GPU-бэкенд (упадёт, если нет CUDA/cuml):
# os.environ["IR_RF_BACKEND"] = "cuml"

!pip install -q -e ".[cuml]"

!ir-pipeline train --paths configs/paths.huggingface.yaml --mode spectrum --config configs/train_mini.yaml --run-dir runs/colab_gpu_rf

### Обучение на обычном железе (только CPU, sklearn)

Подходит для Runtime **без GPU**. Принудительно **`IR_RF_BACKEND=sklearn`**, чтобы не требовалась CUDA и пакет cuML.

Мини-конфиг `train_mini.yaml` и архив `dataset_mini.zip` дают короткий прогон; для полного датасета поменяйте конфиг и `dataset_version` в yaml.

In [ ]:
# Шаг 3b: обучение на CPU (sklearn Random Forest)
import os

os.environ["IR_RF_BACKEND"] = "sklearn"

!ir-pipeline train --paths configs/paths.huggingface.yaml --mode spectrum --config configs/train_mini.yaml --run-dir runs/colab_cpu_rf